In [14]:
%load_ext autoreload
%autoreload 2
import tensorflow as tf

from tensorflow_model_optimization.python.core.keras.compat import keras

import numpy as np

import datetime
from pathlib import Path

from cifar_400k import cifarmodeltest_400k_tf_do50
model = cifarmodeltest_400k_tf_do50()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:

# 1. Load Data
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Shuffle and split Training Data (50,000 total) into Train (45,000) and Val (5,000)

rng = np.random.default_rng(seed=42)
indices = rng.permutation(len(x_train_full))

x_train_full = x_train_full[indices]
y_train_full = y_train_full[indices]

x_train = x_train_full[:45000]
y_train = y_train_full[:45000]
x_val = x_train_full[45000:]
y_val = y_train_full[45000:]

#remove unnecessary dimensions
y_train = y_train.squeeze()
y_val = y_val.squeeze()
y_test = y_test.squeeze()



# Normalize pixel values
x_train = x_train.astype("float32")
x_val = x_val.astype("float32")
x_test = x_test.astype("float32")

# Define Transformations
train_transform = keras.Sequential([
    keras.layers.ZeroPadding2D(padding=4),
    keras.layers.RandomCrop(height=32, width=32),
    keras.layers.RandomFlip("horizontal"),
    keras.layers.Rescaling(scale=1.0 / 128.0, offset=-1.0),

])
val_transform = keras.layers.Rescaling(scale=1.0 / 128.0, offset=-1.0)

BATCH_SIZE = 128

# 3. Create Datasets
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=50000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (train_transform(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .map(lambda x, y: (val_transform(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .map(lambda x, y: (val_transform(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

LOG_DIR = Path("logs") / "fit" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=str(LOG_DIR),
    histogram_freq=1,      # logs weight histograms every epoch
    write_graph=True,
    write_images=False
)

best_model_callback = keras.callbacks.ModelCheckpoint(
    filepath="cifar400k_normalized_dropout50_best.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

# def make_step_decay_schedule(decay_epochs, factor):
#     decay_epochs = set(decay_epochs)

#     def schedule(epoch, lr):
#         if epoch in decay_epochs:
#             return lr * factor
#         return lr

#     return schedule

# lr_callback = keras.callbacks.LearningRateScheduler(
#     make_step_decay_schedule(decay_epochs=[100, 150, 90], factor=0.1),
#     verbose=1
# )




opt = keras.optimizers.Adam(learning_rate=2e-4)
# 4. Compile and Train
model.compile(
    optimizer=opt,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(train_dataset,
          epochs=150,
          validation_data=val_dataset,
          callbacks=[tensorboard_callback,
          best_model_callback])

model.save("cifar400k_normalized_dropout50_test_last.keras")
# 5. Final Evaluation (Independent Test Set)
test_loss, test_acc = model.evaluate(test_dataset, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")


Epoch 1/150


E0000 00:00:1780954452.944654  140402 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape incifarmodeltest_400k_tf/dropout_39/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


349/352 [============================>.] - ETA: 0s - loss: 2.0172 - accuracy: 0.2716
Epoch 1: val_accuracy improved from -inf to 0.44560, saving model to cifar400k_normalized_dropout50_best.keras
352/352 [==============================] - 11s 21ms/step - loss: 2.0160 - accuracy: 0.2717 - val_loss: 1.6901 - val_accuracy: 0.4456
Epoch 2/150
350/352 [============================>.] - ETA: 0s - loss: 1.6788 - accuracy: 0.4223
Epoch 2: val_accuracy improved from 0.44560 to 0.53520, saving model to cifar400k_normalized_dropout50_best.keras
352/352 [==============================] - 7s 19ms/step - loss: 1.6783 - accuracy: 0.4225 - val_loss: 1.4527 - val_accuracy: 0.5352
Epoch 3/150
349/352 [============================>.] - ETA: 0s - loss: 1.4555 - accuracy: 0.5056
Epoch 3: val_accuracy improved from 0.53520 to 0.56660, saving model to cifar400k_normalized_dropout50_best.keras
352/352 [==============================] - 7s 20ms/step - loss: 1.4554 - accuracy: 0.5058 - val_loss: 1.2619 - val_ac

Evaluate best model checkpoint

In [16]:
model_path = "cifar400k_normalized_best.keras"
model = keras.models.load_model(model_path)
test_loss, test_acc = model.evaluate(val_dataset, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")


40/40 - 0s - loss: 0.4919 - accuracy: 0.8908 - 403ms/epoch - 10ms/step

Final Test Accuracy: 0.8908


In [ ]:
# Regular post-training quantization (PTQ) from the trained float model.
# This does not run QAT; it only calibrates int8 ranges using representative data.
from cifar_400k import (
    cifarmodeltest_400k_tf_do50_functional,
    copy_cifarmodeltest_400k_weights,
)

ptq_float_model = cifarmodeltest_400k_tf_do50_functional()
copy_cifarmodeltest_400k_weights(model, ptq_float_model)


def ptq_representative_data_gen():
    for images, _ in val_dataset.unbatch().batch(1).take(100):
        yield [images]


ptq_converter = tf.lite.TFLiteConverter.from_keras_model(ptq_float_model)
ptq_converter.optimizations = [tf.lite.Optimize.DEFAULT]
ptq_converter.representative_dataset = ptq_representative_data_gen
ptq_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
ptq_converter.inference_input_type = tf.int8
ptq_converter.inference_output_type = tf.int8

print("Converting float model with regular PTQ...")
ptq_tflite_model = ptq_converter.convert()
ptq_tflite_output_path = "cifar400k_ptq_int8.tflite"
with open(ptq_tflite_output_path, "wb") as f:
    f.write(ptq_tflite_model)

print(f"Success! PTQ int8 model saved to: {ptq_tflite_output_path}")

Train again with QAT

In [19]:
import datetime
from pathlib import Path
import tensorflow_model_optimization as tfmot
from cifar_400k import (
    cifarmodeltest_400k_tf_do50_functional,
    copy_cifarmodeltest_400k_weights,
)

quantize_model = tfmot.quantization.keras.quantize_model

# TFMOT requires a Functional or Sequential graph. The source model is subclassed,
# so build the same residual graph functionally and copy the trained weights into it.
functional_model = cifarmodeltest_400k_tf_do50_functional()
copy_cifarmodeltest_400k_weights(model, functional_model)

# q_aware stands for quantization aware.
q_aware_model = quantize_model(functional_model)

# TensorBoard logs for QAT run
QAT_LOG_DIR = Path("logs") / "fit" / ("qat-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

qat_tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=str(QAT_LOG_DIR),
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

# `quantize_model` requires a recompile.
q_aware_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

q_aware_model.summary()

ValueError: `to_quantize` can only either be a keras Sequential or Functional model.

In [ ]:
q_aware_model.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    callbacks=[qat_tensorboard_callback,
    best_model_callback]
)
q_aware_model.save("./cifartest_QAT.keras")


Convert to TFLITE

In [ ]:
print("Initializing TFLite Converter...")
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = ptq_representative_data_gen

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print("Quantizing model...")
quantized_tflite_model = converter.convert()
tflite_output_path = "q8_model.tflite"
with open(tflite_output_path, "wb") as f:
    f.write(quantized_tflite_model)

print(f"Success! Full int8 model saved to: {tflite_output_path}")

In [ ]:
# Compare regular float, QAT, and PTQ performance on the test set.
import os
import numpy as np


def evaluate_tflite_classifier(model_path, dataset):
    if not os.path.exists(model_path):
        return None

    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    accuracy = keras.metrics.SparseCategoricalAccuracy()
    loss_metric = keras.metrics.Mean()

    for images, labels in dataset.unbatch().batch(1):
        input_data = images.numpy()
        if np.issubdtype(input_details["dtype"], np.integer):
            input_data = input_data / input_scale + input_zero_point
            input_data = np.clip(
                np.rint(input_data),
                np.iinfo(input_details["dtype"]).min,
                np.iinfo(input_details["dtype"]).max,
            ).astype(input_details["dtype"])
        else:
            input_data = input_data.astype(input_details["dtype"])

        interpreter.set_tensor(input_details["index"], input_data)
        interpreter.invoke()

        logits = interpreter.get_tensor(output_details["index"])
        if np.issubdtype(output_details["dtype"], np.integer):
            logits = (logits.astype(np.float32) - output_zero_point) * output_scale

        labels = tf.reshape(labels, [-1])
        batch_loss = keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        loss_metric.update_state(batch_loss)
        accuracy.update_state(labels, logits)

    return float(loss_metric.result().numpy()), float(accuracy.result().numpy())


results = []

regular_loss, regular_acc = model.evaluate(test_dataset, verbose=0)
results.append(("Regular float Keras", regular_loss, regular_acc))

if "q_aware_model" in globals():
    qat_loss, qat_acc = q_aware_model.evaluate(test_dataset, verbose=0)
    results.append(("QAT fake-quant Keras", qat_loss, qat_acc))

qat_tflite_metrics = evaluate_tflite_classifier("q8_model.tflite", test_dataset)
if qat_tflite_metrics is not None:
    results.append(("QAT int8 TFLite", *qat_tflite_metrics))
else:
    print("Skipping QAT int8 TFLite: q8_model.tflite not found. Run the QAT conversion cell first.")

ptq_tflite_metrics = evaluate_tflite_classifier("cifar400k_ptq_int8.tflite", test_dataset)
if ptq_tflite_metrics is not None:
    results.append(("PTQ int8 TFLite", *ptq_tflite_metrics))
else:
    print("Skipping PTQ int8 TFLite: cifar400k_ptq_int8.tflite not found. Run the PTQ conversion cell first.")

print("\nTest-set comparison")
print(f"{'Model':<24} {'Loss':>10} {'Accuracy':>10}")
print("-" * 48)
for name, loss, acc in results:
    print(f"{name:<24} {loss:>10.4f} {acc:>10.4f}")